# Model 3b: SARIMAX + Exogenous Calendar Features for Drug `M01AB`

## Workflow Checklist:
1. **Pre-Training Suitability Checks**: Construct calendar exogenous matrix ($X_t$) & test for multicollinearity (VIF).
2. **Exogenous Feature Integration**: Incorporate DayOfWeek, Month, and Weekend indicators.
3. **Fit on TRAIN (2014–2017)**: Estimate SARIMAX coefficients.
4. **Validate on 2018 Validation**: Evaluate Val RMSLE.
5. **Refit & Forecast 2019 Test**: Evaluate final holdout metrics.


In [1]:
import os
import sys
import warnings
import subprocess

# Auto-Dependency Guard: Check and install missing packages dynamically
pkg_map = {
    'prophet': 'prophet',
    'statsmodels': 'statsmodels',
    'lightgbm': 'lightgbm',
    'xgboost': 'xgboost',
    'shap': 'shap',
    'torch': 'torch',
    'sklearn': 'scikit-learn',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn'
}

missing = []
for mod_name, pip_name in pkg_map.items():
    try:
        __import__(mod_name)
    except ImportError:
        missing.append(pip_name)

if missing:
    print(f"Installing missing dependencies: {missing}...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
        print("Dependencies successfully installed!")
    except Exception as err:
        print(f"Warning: Auto-pip install notice ({err}). Proceeding with environment packages...")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

TARGET_DRUG = 'M01AB'

# Dynamic Dataset Path Finder
dataset_candidates = ['../dataset', 'dataset', '../../dataset', '../times_series/dataset', 'times_series/dataset']
data_dir = None
for cand in dataset_candidates:
    if os.path.exists(os.path.join(cand, 'train_daily.csv')):
        data_dir = cand
        break

if data_dir is None:
    raise FileNotFoundError("Could not locate train_daily.csv dataset")

train_df = pd.read_csv(os.path.join(data_dir, 'train_daily.csv'))
val_df = pd.read_csv(os.path.join(data_dir, 'val_daily.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test_daily.csv'))

for df in [train_df, val_df, test_df]:
    df['date'] = pd.to_datetime(df['date'])

train_series = train_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
val_series = val_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
test_series = test_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')

combined_series = pd.concat([train_series, val_series]).asfreq('D')
full_series = pd.concat([combined_series, test_series]).asfreq('D')

def evaluate_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.clip(np.array(y_pred), 0, None)
    
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mae = np.mean(np.abs(y_true - y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1, y_true))) * 100
    wape = (np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)) * 100
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))
    return {'RMSLE': rmsle, 'RMSE': rmse, 'MAE': mae, 'MAPE (%)': mape, 'WAPE (%)': wape}

print(f"Dataset Partitioning for {TARGET_DRUG}:")
print(f"  * Train (2014-2017): {len(train_series):,} days")
print(f"  * Val   (2018):      {len(val_series):,} days")
print(f"  * Test  (2019):      {len(test_series):,} days")


Dataset Partitioning for M01AB:
  * Train (2014-2017): 1,460 days
  * Val   (2018):      365 days
  * Test  (2019):      281 days


In [2]:
# Step 1: Pre-Training Suitability Checks (Exogenous Feature Construction)
all_dates = pd.date_range('2014-01-01', '2019-12-31')
ex_all = pd.DataFrame(index=all_dates)
ex_all['dayofweek'] = all_dates.dayofweek
ex_all['month'] = all_dates.month
ex_all['is_weekend'] = (all_dates.dayofweek >= 5).astype(float)
ex_all_dummies = pd.get_dummies(ex_all, columns=['dayofweek', 'month'], drop_first=True).astype(float)

def make_exog(df_idx):
    return ex_all_dummies.loc[df_idx]

exog_tr = make_exog(train_series.index)
exog_va = make_exog(val_series.index)
exog_cb = make_exog(combined_series.index)
exog_ts = make_exog(test_series.index)

print(f"Exogenous Feature Matrix Shape: {exog_tr.shape} (Dummies for DayOfWeek & Month)")


Exogenous Feature Matrix Shape: (1460, 18) (Dummies for DayOfWeek & Month)


In [3]:
# Step 2: Fit SARIMAX(1,1,1)x(1,0,1)_7 + Exog on TRAIN & Validate on 2018
from statsmodels.tsa.statespace.sarimax import SARIMAX

train_y_log = np.log1p(train_series)
m3_tr = SARIMAX(train_y_log, order=(1,1,1), seasonal_order=(1,0,1,7), exog=exog_tr, enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

val_pred_log = m3_tr.get_forecast(steps=len(val_series), exog=exog_va).predicted_mean
m3_val_pred = np.clip(np.expm1(val_pred_log), 0, None)
val_metrics = evaluate_metrics(val_series, m3_val_pred)

print(f"Validation Metrics (2018) for SARIMAX + Exog:")
for k, v in val_metrics.items():
    print(f"  * {k:10s}: {v:.4f}")


Validation Metrics (2018) for SARIMAX + Exog:
  * RMSLE     : 0.5357
  * RMSE      : 2.7976
  * MAE       : 2.1968
  * MAPE (%)  : 75.0701
  * WAPE (%)  : 44.8711


C:\Users\ranje\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [4]:
# Step 3: Refit on Train+Val & Forecast 2019 Test Holdout
comb_y_log = np.log1p(combined_series)
m3_full = SARIMAX(comb_y_log, order=(1,1,1), seasonal_order=(1,0,1,7), exog=exog_cb, enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

test_pred_log = m3_full.get_forecast(steps=len(test_series), exog=exog_ts).predicted_mean
m3_test_pred = np.clip(np.expm1(test_pred_log), 0, None)
test_metrics = evaluate_metrics(test_series, m3_test_pred)

print(f"=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 3b: SARIMAX + EXOG ===")
for k, v in test_metrics.items():
    print(f"  * {k:10s}: {v:.4f}")

pd.DataFrame({'date': test_series.index, 'pred_SARIMAX': m3_test_pred.values}).to_csv('m3_sarimax_preds.csv', index=False)


=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 3b: SARIMAX + EXOG ===
  * RMSLE     : 0.5091
  * RMSE      : 3.0107
  * MAE       : 2.3062
  * MAPE (%)  : 66.1619
  * WAPE (%)  : 42.7105


C:\Users\ranje\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
